# Sesión 4. Interpretación y aplicaciones

En las sesiones anteriores aprendimos a medir ancestría: global con ADMIXTURE, local con RFMix. Esta sesión es sobre **para qué sirve**.

Vamos a trabajar sobre los resultados ya calculados de las 345 personas de CLM, MXL, PUR y PEL, y a responder tres preguntas:

1. ¿Qué ancestría tiene una persona **en el lugar exacto** donde está una variante de riesgo?
2. ¿Cómo se buscan regiones del genoma asociadas a una enfermedad usando la ancestría? (*admixture mapping*)
3. ¿Por qué los puntajes de riesgo poligénico (PRS) funcionan peor en poblaciones mezcladas?

A diferencia de las otras sesiones, aquí hay poco código nuevo y mucha interpretación.

## 1. Cargar los resultados

Son los mismos archivos de la sesión 3, pero con las 345 personas de las cuatro poblaciones.

In [ ]:
%%bash
for f in rfmix_todos.msp.tsv.gz rfmix_todos.rfmix.Q poblaciones.tsv; do
  wget -nv -nc https://github.com/YTZ27/curso-ancestria/raw/main/data/ancestria_local/$f
done
ls -lh

In [ ]:
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

msp = pd.read_csv('rfmix_todos.msp.tsv.gz', sep='\t', skiprows=1)
pob = pd.read_csv('poblaciones.tsv', sep='\t')

# La primera línea del .msp.tsv dice qué número corresponde a cada ancestría.
# El archivo está comprimido, así que lo abrimos con gzip para leer esa línea.
with gzip.open('rfmix_todos.msp.tsv.gz', 'rt') as f:
    primera_linea = f.readline()
codigos = {int(c): a for a, c in (x.split('=') for x in primera_linea.split(':')[1].split())}
haplotipos = list(msp.columns[6:])

# A qué población pertenece cada haplotipo
haplo_pob = pd.Series({h: h.rsplit('.', 1)[0] for h in haplotipos}).map(pob.set_index('muestra')['poblacion'])

colores = {'AFR': '#eb6834', 'EUR': '#2a78d6', 'NAT': '#1baf7a'}
poblaciones = ['PUR', 'CLM', 'MXL', 'PEL']

print(codigos)
print(f'{len(msp)} segmentos, {len(haplotipos)} haplotipos')
haplo_pob.value_counts()

## 2. Ancestría local en una posición concreta

### El caso de APOL1

El gen **APOL1** está en el cromosoma 22 (posiciones 36 253 071 a 36 267 530 en GRCh38). Dos variantes suyas, llamadas **G1** y **G2**, aumentan fuertemente el riesgo de enfermedad renal crónica cuando una persona tiene dos copias de riesgo (Tzur et al., 2010).

Lo interesante para nosotros es de dónde vienen esas variantes:

- Prácticamente **solo existen en haplotipos de origen africano occidental**. Su frecuencia en poblaciones europeas es cercana a cero.
- Se cree que se mantuvieron porque protegen contra el tripanosoma, el parásito que causa la enfermedad del sueño.
- El locus se **descubrió por admixture mapping** en el cromosoma 22: se vio que las personas afroamericanas con enfermedad renal tenían más ancestría africana en esa región que en el resto del genoma.

La consecuencia clínica es directa: el riesgo que aporta APOL1 depende de **cuántos de los dos haplotipos de la persona son africanos en esa posición**, no de su ancestría africana global.

Busquemos el segmento que contiene al gen.

In [ ]:
INICIO, FIN = 36253071, 36267530   # APOL1 en GRCh38

segmento = msp[(msp['spos'] <= INICIO) & (msp['epos'] >= FIN)]
segmento[['#chm', 'spos', 'epos', 'n snps']]

Ese segmento abarca todo el gen. Ahora miramos qué ancestría tiene cada haplotipo ahí, y lo resumimos por población.

In [ ]:
fila = segmento.iloc[0]
ancestria_apol1 = pd.Series({h: codigos[fila[h]] for h in haplotipos})

tabla = (pd.crosstab(haplo_pob, ancestria_apol1, normalize='index')
           .reindex(index=poblaciones, columns=list(colores), fill_value=0)
           .rename_axis(index='poblacion', columns='ancestria en APOL1'))
(tabla * 100).round(1)

Esa tabla dice, para cada población, qué porcentaje de los haplotipos es africano, europeo o nativo americano **en la posición de APOL1**.

Pero lo que importa clínicamente no es el haplotipo suelto, sino la persona: ¿cuántas de sus dos copias son africanas ahí? Porque el riesgo de APOL1 es **recesivo**: hace falta tener dos copias de riesgo.

In [ ]:
# Para cada persona, cuántos de sus 2 haplotipos son africanos en APOL1
muestras = sorted({h.rsplit('.', 1)[0] for h in haplotipos})
copias = pd.Series({m: sum(ancestria_apol1[f'{m}.{k}'] == 'AFR' for k in (0, 1)) for m in muestras})
copias_pob = pd.DataFrame({'copias_africanas': copias}).join(pob.set_index('muestra')['poblacion'])

resumen = (pd.crosstab(copias_pob['poblacion'], copias_pob['copias_africanas'], normalize='index')
             .reindex(poblaciones) * 100).round(1)
resumen.columns = [f'{c} copias' if c != 1 else '1 copia' for c in resumen.columns]
resumen.rename_axis(index='poblacion', columns='copias africanas en APOL1')

In [ ]:
ax = resumen.plot.bar(stacked=True, figsize=(7, 4), width=0.7,
                      color=['#d9d9d6', '#f0a882', colores['AFR']], edgecolor='none')
ax.set(ylim=(0, 100), xlabel='', ylabel='% de personas',
       title='Copias de ancestría africana en la posición de APOL1')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='', bbox_to_anchor=(1, 1), frameon=False)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

**Para discutir en clase:**

- Solo las personas con **dos** copias africanas en esa posición pueden ser homocigotas para G1 o G2. En las demás, el genotipo de riesgo recesivo es imposible por ancestría local.
- Tener dos copias africanas ahí **no** significa tener la variante de riesgo: la frecuencia de G1 en poblaciones africanas occidentales es de alrededor del 40 %, no del 100 %. La ancestría local acota quién *puede* tenerla.
- Compara la tabla con la ancestría africana **global** de cada población (sesión 3). ¿Una persona con 10 % de ancestría africana global puede tener dos copias africanas en APOL1?

**Ejercicio:** cambia `INICIO` y `FIN` por otra región del cromosoma 22 que te interese y vuelve a correr las tres celdas anteriores.

## 3. Admixture mapping

### La idea

En una población mezclada, cada persona tiene una proporción de ancestría global, pero a lo largo del genoma esa proporción varía por azar. Si una enfermedad es más frecuente en uno de los grupos ancestrales por razones **genéticas**, entonces las personas enfermas deberían tener, en la región responsable, **más ancestría de ese grupo** que en el resto de su genoma.

Eso es admixture mapping: en lugar de buscar variante por variante como en un GWAS, se busca **exceso de ancestría local** en los casos. Necesita muchos menos marcadores que un GWAS, porque los bloques de ancestría son grandes.

### Lo que podemos hacer aquí

No tenemos casos ni controles, así que no podemos hacer un admixture mapping de verdad. Lo que sí podemos es ver **la mitad del método**: cómo se mide la ancestría local a lo largo del cromosoma y cuánto se desvía del promedio.

In [ ]:
centro = (msp['spos'] + msp['epos']) / 2 / 1e6

# Para cada población y cada segmento: fracción de haplotipos de cada ancestría
fracciones = {}
for p in poblaciones:
    cols = haplo_pob[haplo_pob == p].index
    fracciones[p] = pd.DataFrame({nombre: (msp[cols] == cod).mean(axis=1)
                                  for cod, nombre in codigos.items()})

fracciones['CLM'].describe().round(3)

La desviación respecto al promedio de cada población, en puntos porcentuales:

In [ ]:
fig, axes = plt.subplots(len(poblaciones), 1, figsize=(10, 9), sharex=True, sharey=True)
for ax, p in zip(axes, poblaciones):
    f = fracciones[p]
    for nombre in colores:
        ax.plot(centro, (f[nombre] - f[nombre].mean()) * 100, color=colores[nombre], label=nombre, linewidth=1.3)
    ax.axhline(0, color='#999999', linewidth=0.8, zorder=0)
    ax.set_title(p, loc='left')
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)
axes[0].legend(title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
axes[-1].set_xlabel('Posición en el cromosoma 22 (Mb)')
fig.supylabel('Desviación respecto al promedio de la población (puntos porcentuales)')
plt.tight_layout()
plt.show()

**Cómo leerlo:** si la ancestría se repartiera de forma perfectamente uniforme, las líneas serían planas en cero. Las ondulaciones que ves son normales: son fluctuaciones por azar, porque cada persona heredó bloques grandes de ancestría y con unas decenas de personas esos bloques no se promedian del todo.

En un admixture mapping real, la pregunta no es si una región se desvía, sino si **se desvía más en los casos que en los controles**, y hay que corregir por la cantidad de regiones evaluadas.

¿Qué tan grande es la desviación máxima en cada población?

In [ ]:
pd.DataFrame({p: {nombre: round((fracciones[p][nombre] - fracciones[p][nombre].mean()).abs().max() * 100, 1)
                  for nombre in colores} for p in poblaciones}).T

**Advertencia importante:** estas desviaciones son de un solo cromosoma y de unas pocas decenas de personas por población. No alcanzan para concluir nada sobre selección ni sobre enfermedad. Un estudio real usa todo el genoma, miles de personas y un diseño de casos y controles.

**Para discutir:** ¿por qué el admixture mapping necesita muchos menos marcadores que un GWAS? Pista: piensa en el tamaño de los bloques del karyograma de la sesión 3, comparado con el tamaño de un bloque de desequilibrio de ligamiento dentro de una población.

## 4. Ancestría y puntajes de riesgo poligénico (PRS)

Un PRS suma el efecto de muchas variantes para estimar el riesgo de una persona. Los pesos de esa suma salen de un GWAS, y la gran mayoría de los GWAS se han hecho en poblaciones de ancestría europea.

Cuando ese PRS se aplica a una persona de otra ancestría, pierde precisión, por tres razones:

| Razón | Qué pasa |
|---|---|
| **Frecuencias distintas** | Una variante común en europeos puede ser rara en otra población, así que aporta menos información |
| **Patrones de LD distintos** | El GWAS casi nunca encuentra la variante causal, sino una vecina correlacionada. Esa correlación cambia entre poblaciones, así que el peso deja de aplicar |
| **Variantes ausentes** | Variantes importantes en una población pueden no existir en la otra, y el GWAS europeo ni siquiera las midió |

El resultado es que un PRS derivado de europeos pierde buena parte de su capacidad predictiva en personas de ancestría africana, y queda en un punto intermedio en poblaciones mezcladas (Martin et al., 2019).

**La conexión con esta sesión:** en una persona mezclada, el problema no es uniforme a lo largo del genoma. El PRS funciona mejor en los tramos de ancestría europea y peor en los demás. Por eso hay métodos que incorporan la ancestría local al calcular el puntaje.

**Para discutir:** si tuvieras que aplicar un PRS de riesgo cardiovascular en una cohorte colombiana, ¿qué harías antes de reportar resultados individuales?

## 5. Cierre: qué aprendimos y qué falta

Lo que podemos hacer con lo visto en el curso:

- Control de calidad de datos genómicos y entender sus formatos (sesión 1)
- Estimar ancestría global con PCA y ADMIXTURE, e interpretarla con cuidado (sesión 2)
- Estimar ancestría local con RFMix y visualizarla (sesión 3)
- Consultar la ancestría en una posición concreta y entender la lógica del admixture mapping (sesión 4)

Lo que no alcanzamos a cubrir y vale la pena si sigues por este camino:

| Tema | Por qué importa |
|---|---|
| Elegir K con validación cruzada (`--cv` en ADMIXTURE) | Para no fijar K a dedo |
| Phasing con SHAPEIT5 o Beagle | Usamos datos ya faseados; con datos propios hay que hacerlo |
| Todo el genoma, no un cromosoma | Las estimaciones de ancestría global de un solo cromosoma son imprecisas |
| Modo supervisado de ADMIXTURE | Fija las poblaciones de referencia en lugar de dejar que el modelo las encuentre |
| Fechar la mezcla | Del largo de los segmentos se estima hace cuántas generaciones ocurrió (Gravel, 2012) |

## Referencias

- Gravel S. Population genetics models of local ancestry. *Genetics*. 2012;191:607–619. doi:10.1534/genetics.112.139808
- Martin AR, Kanai M, Kamatani Y, Okada Y, Neale BM, Daly MJ. Clinical use of current polygenic risk scores may exacerbate health disparities. *Nat Genet*. 2019;51:584–591. doi:10.1038/s41588-019-0379-x
- Tzur S, Rosset S, Shemer R, et al. Missense mutations in the APOL1 gene are highly associated with end stage kidney disease risk previously attributed to the MYH9 gene. *Hum Genet*. 2010;128(3):345–350. doi:10.1007/s00439-010-0861-0
- Maples BK, Gravel S, Kenny EE, Bustamante CD. RFMix: a discriminative modeling approach for rapid and robust local-ancestry inference. *Am J Hum Genet*. 2013;93:278–288. doi:10.1016/j.ajhg.2013.06.020